In [1]:
# Install required packages inside the notebook
%pip install anomalib matplotlib numpy opencv-python torch

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 851.8/851.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.8/241.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.5/760.5 kB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 64.3 MB/s eta 0:00:00
  Created wheel for freia: filename=FrEIA-0.2-py3-none-any.whl size=42763 sha256=36c3bad707248991f33bd2b18658edac408b399f9c1cc3986d469cad57a76226
  Stored in d

In [2]:
!git clone https://github.com/Ap4chee/hacknationAnomalieFixed

Cloning into 'hacknationAnomalieFixed'...
remote: Enumerating objects: 26436, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 26436 (delta 8), reused 25 (delta 7), pack-reused 26403 (from 2)
Receiving objects: 100% (26436/26436), 985.31 MiB | 30.60 MiB/s, done.
Resolving deltas: 100% (3355/3355), done.
Updating files: 100% (295/295), done.


# Data Normalization

In [3]:
import os
import sys

# Ensure the current directory is in the path so local modules can be imported
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from hacknationAnomalieFixed.ImageProcessor.XRayImage import XRayImage

def normalize(data_folder: str, output_directory: str):

    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    if not os.path.exists(data_folder):
        print(f"Folder ze zdjęciami do przetworzenia nie istnieje: {data_folder}")
        return


    for root, dirs, files in os.walk(data_folder):
        for file in files:
            if file.find('czarno') == -1:
                continue

            # Sprawdzamy czy to obrazek
            if file.lower().endswith('.bmp'):
                full_path = os.path.join(root, file)

                img = XRayImage(src=full_path)
                img.applyFilters()
                img.generateTiles()
                img.saveTiles(output_directory)

print("Data normalization functions defined.")

Data normalization functions defined.


In [4]:
# Run normalization
normalize("hacknationAnomalieFixed/raw_data/czyste", "datasets/train/good")
normalize("hacknationAnomalieFixed/raw_data/brudne", "datasets/test/defect")
print("Normalization finished.")

Normalization finished.


# Training with Anomalib

In [ ]:
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

def train_model():
    # 1. Konfiguracja Danych
    datamodule = Folder(
        name="datasets",
        root="datasets",
        normal_dir="train/good",   # Folder treningowy
        abnormal_dir="test/defect", # Folder z defektami do testów
        #normal_test_dir="test/good", # Folder z dobrymi próbkami do testów
        train_batch_size=32,
        eval_batch_size=32,
        # WAŻNE NA WINDOWS: Czasem warto ustawić num_workers na 0, jeśli nadal będą błędy
        num_workers=2,
    )

    # setup() też warto wywołać wewnątrz main
    datamodule.setup()

    # 2. Inicjalizacja Modelu
    model = Patchcore(
        backbone="resnet18",
        pre_trained=True
    )

    # 3. Konfiguracja Silnika (Engine)
    engine = Engine(
        accelerator="auto",
        devices=1,
        max_epochs=1,
        enable_progress_bar=False,  # KLUCZOWE: Wyłącza problematyczny pasek Rich
    )

    # 4. Trening
    print("Rozpoczynam trening...")
    engine.fit(datamodule=datamodule, model=model)

    # 5. Testowanie
    print("Rozpoczynam testy...")
    # Tutaj poprawka z poprzedniej odpowiedzi (setup dla testu), aby uniknąć błędu iter()
    datamodule.setup(stage="test")
    test_results = engine.test(datamodule=datamodule, model=model)
    print(test_results)


In [ ]:
if __name__ == "__main__":
    # Ten blok jest KLUCZOWY na Windowsie
    train_model()

INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Rozpoczynam trening...


Exception ignored in: <function tqdm.__del__ at 0x7dd4ced63060>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1277, in close
    if self.last_print_t < self.start_t + self.delay:
       ^^^^^^^^^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'last_print_t'


Selecting Coreset Indices.:   0%|          | 0/25180 [00:00<?, ?it/s]

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │  2.8 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:534: Found 69 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
Selecting Coreset Indices.:  51%|█████▏    | 129223/251801 [1:37:28<1:32:27, 22.10it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')